In [1]:
#Importing and Setting up SQL

import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [2]:
cursor.executescript("""

CREATE TABLE Customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT
);


CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date DATE,
    amount REAL,
    FOREIGN KEY(customer_id) REFERENCES Customers(customer_id)
);


CREATE TABLE Products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT
);


CREATE TABLE Order_Items (
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    FOREIGN KEY(order_id) REFERENCES Orders(order_id),
    FOREIGN KEY(product_id) REFERENCES Products(product_id)
);

""")

conn.commit()

In [3]:
cursor.executescript("""

INSERT INTO Customers VALUES
(1,'John','New York'),
(2,'Alice','London'),
(3,'Bob','Paris');


INSERT INTO Orders VALUES
(101,1,'2025-01-10',250),
(102,1,'2025-02-15',400),
(103,2,'2025-02-20',300),
(104,3,'2025-03-01',150);


INSERT INTO Products VALUES
(1,'Laptop','Electronics'),
(2,'Phone','Electronics'),
(3,'Chair','Furniture');


INSERT INTO Order_Items VALUES
(101,1,1),
(101,2,2),
(102,1,1),
(103,3,4),
(104,2,1);

""")

conn.commit()

In [4]:
def run_query(query):
    return pd.read_sql_query(query, conn)

# **INNER JOIN**

In [5]:
run_query("""
SELECT
    c.name,
    o.order_id,
    o.amount
FROM Customers c
JOIN Orders o
ON c.customer_id=o.customer_id;
""")

,name,order_id,amount
0,John,101,250.0
1,John,102,400.0
2,Alice,103,300.0
3,Bob,104,150.0


# **MULTIPLE TABLE JOIN**

In [6]:
run_query("""
SELECT
    c.name,
    p.product_name,
    oi.quantity
FROM Customers c

JOIN Orders o
ON c.customer_id=o.customer_id

JOIN Order_Items oi
ON o.order_id=oi.order_id

JOIN Products p
ON oi.product_id=p.product_id;
""")

,name,product_name,quantity
0,John,Laptop,1
1,John,Phone,2
2,John,Laptop,1
3,Alice,Chair,4
4,Bob,Phone,1


# **WINDOW FUNCTIONS**

In [7]:
run_query("""
SELECT
    customer_id,
    name,
    total_spent,

    RANK() OVER(
        ORDER BY total_spent DESC
    ) AS rank

FROM
(
SELECT
    c.customer_id,
    c.name,
    SUM(o.amount) AS total_spent

FROM Customers c

JOIN Orders o
ON c.customer_id=o.customer_id

GROUP BY c.customer_id
);
""")

,customer_id,name,total_spent,rank
0,1,John,650.0,1
1,2,Alice,300.0,2
2,3,Bob,150.0,3


In [8]:
run_query("""
SELECT
    order_id,
    order_date,
    amount,

    SUM(amount)
    OVER(
        ORDER BY order_date
    ) AS running_total

FROM Orders;
""")

,order_id,order_date,amount,running_total
0,101,2025-01-10,250.0,250.0
1,102,2025-02-15,400.0,650.0
2,103,2025-02-20,300.0,950.0
3,104,2025-03-01,150.0,1100.0


In [9]:
run_query("""
SELECT
    order_id,
    amount,

    LAG(amount)
    OVER(
        ORDER BY order_date
    ) AS previous_amount

FROM Orders;
""")

,order_id,amount,previous_amount
0,101,250.0,NaN
1,102,400.0,250.0
2,103,300.0,400.0
3,104,150.0,300.0
